# CO-SPY Fusion V3 — Kaggle GPU Verified

Notebook này **chỉ train Fusion V3 từ các checkpoint Semantic + Artifact + NPR có sẵn**; không train lại ba nhánh.

## Kaggle Input cần có

- `Co-Spy_v1`
  - `mscoco2017`
  - `stable-diffusion-v1-4`
- `Co-spy_v2`
  - `ckpt/sd-v1_4/semantic/checkpoint_best.pth`
  - `ckpt/sd-v1_4/artifact/checkpoint_best.pth`
  - `ckpt/sd-v1_4/npr/checkpoint_best.pth`

## Accelerator

Bật **GPU T4 x2** trong Kaggle trước khi chạy.

> Lưu ý: CO-SPY hiện được chạy rõ ràng trên `cuda:0`. Kaggle có 2 T4 nhưng việc có 2 GPU không tự động làm code dùng cả hai.
>
> Nếu xuất hiện warning kiểu `Could not find cuda drivers ... GPU will not be used` từ TensorFlow/JAX nhưng cell kiểm tra bên dưới báo `PyTorch CUDA: True` và phép tính thật trên GPU chạy thành công, thì CO-SPY/PyTorch vẫn dùng GPU bình thường.


In [1]:
# 1. GPU SANITY CHECK — chạy trước mọi thứ khác

import os
import subprocess
import torch

# Giảm log phụ từ các framework khác; không ảnh hưởng PyTorch.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("=== NVIDIA SMI ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== PYTORCH CUDA ===")
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch không nhìn thấy CUDA. Hãy bật Kaggle Accelerator = GPU rồi restart session."
    )

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

GPU_ID = 0
torch.cuda.set_device(GPU_ID)
DEVICE = torch.device(f"cuda:{GPU_ID}")

print(f"\n=== REAL CUDA COMPUTE TEST trên {DEVICE} ===")
a = torch.randn((4096, 4096), device=DEVICE, dtype=torch.float16)
b = torch.randn((4096, 4096), device=DEVICE, dtype=torch.float16)
torch.cuda.synchronize()
c = a @ b
torch.cuda.synchronize()

print("Tensor device:", c.device)
print("Result shape:", tuple(c.shape))
print("Allocated VRAM:", round(torch.cuda.memory_allocated(GPU_ID) / 1024**2, 2), "MiB")
print("Peak VRAM:", round(torch.cuda.max_memory_allocated(GPU_ID) / 1024**2, 2), "MiB")
print("\n✅ PYTORCH CUDA TEST PASSED — CO-SPY có thể dùng GPU.")

del a, b, c
torch.cuda.empty_cache()


=== NVIDIA SMI ===
Thu Nov 19 18:08:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

In [2]:
# 2. Clone repo + áp dụng patch Fusion V3 nếu repo trên GitHub còn commit cũ

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path("/kaggle/working/CO-SPY")

if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/datnguyentrung/CO-SPY.git", str(REPO)],
        check=True,
    )

os.chdir(REPO)

print("Repository:", REPO)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip(),
)

# Tự áp dụng patch nếu GitHub còn commit cũ.
fusion_file = REPO / "detectors" / "sd_v1_4" / "fusion.py"
fusion = fusion_file.read_text(encoding="utf-8")

fusion = fusion.replace(
    "feature_dims = [self.sem.fc.in_features, self.art.fc.in_features]",
    "feature_dims = [self.sem.fc.in_features + self.sem.fc.out_features, self.art.fc.in_features]",
)

fusion = fusion.replace(
    "sem_feat = self.sem.clip.encode_image(x_sem)",
    "            sem_embedding = self.sem.clip.encode_image(x_sem)\n"
    "            sem_logit = self.sem.fc(sem_embedding)\n"
    "            sem_feat = torch.cat([sem_embedding, sem_logit], dim=1)",
)

fusion_file.write_text(fusion, encoding="utf-8")

train_file = REPO / "train.py"
train = train_file.read_text(encoding="utf-8")

if "semantic_feature_schema=clip_plus_trained_logit_v1" not in train:
    train = train.replace(
        "            self.cache_key = hashlib.sha256(",
        '            if self.fusion_head != "v1":\n'
        '                signature += "|semantic_feature_schema=clip_plus_trained_logit_v1"\n'
        "            self.cache_key = hashlib.sha256(",
    )
    train_file.write_text(train, encoding="utf-8")

assert "sem_logit = self.sem.fc(sem_embedding)" in fusion_file.read_text(encoding="utf-8")

print("Fusion V3 patch: OK")


Cloning into '/kaggle/working/CO-SPY'...


Repository: /kaggle/working/CO-SPY
Commit: 85b2c72
Fusion V3 patch: OK


In [3]:
# 3. Cài dependencies

packages = [
    "diffusers==0.32.2",
    "open-clip-torch==2.31.0",
    "timm==1.0.15",
    "loguru==0.7.3",
    "accelerate==1.0.1",
    "ftfy==6.2.3",
    "transformers==4.46.3",
    "safetensors==0.5.3",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Dependencies installed")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
urwid 5.1.3 requires wcwidth!=0.9.0,!=0.9.1,>=0.4, but you have wcwidth 0.2.14 which is incompatible.
litellm 1.109.1 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
prettytable 3.18.0 requires wcwidth>=0.3.5, but you have wcwidth 0.2.14 which is incompatible.


Dependencies installed


In [4]:
# 4. Tìm dataset + checkpoint trong /kaggle/input và chuẩn bị cấu trúc cho repo

from pathlib import Path
import shutil

INPUT = Path("/kaggle/input")

# -------- Dataset Co-Spy_v1 --------
dataset_candidates = [
    p
    for p in INPUT.rglob("*")
    if p.is_dir()
    and (p / "mscoco2017").is_dir()
    and (p / "stable-diffusion-v1-4").is_dir()
]

if not dataset_candidates:
    raise FileNotFoundError(
        "Không tìm thấy dataset Co-Spy_v1 chứa cả mscoco2017 và stable-diffusion-v1-4."
    )

data_root = dataset_candidates[0]

# -------- Checkpoint Co-spy_v2 --------
# Repo CO-SPY của bạn dùng tên sd-v1_4.
# Để notebook bền hơn, vẫn chấp nhận cả sd-v1_4 và sd-v1-4.
CKPT_DATASET_DIR_NAMES = ["sd-v1_4", "sd-v1-4"]

ckpt_candidates = []

for ckpt_dir in INPUT.rglob("ckpt"):
    if not ckpt_dir.is_dir():
        continue

    for dataset_dir_name in CKPT_DATASET_DIR_NAMES:
        base = ckpt_dir / dataset_dir_name

        if not base.is_dir():
            continue

        found = {}
        for branch in ["semantic", "artifact", "npr"]:
            matches = list((base / branch).rglob("checkpoint_best.pth"))
            found[branch] = matches

        if all(found[branch] for branch in ["semantic", "artifact", "npr"]):
            ckpt_candidates.append(
                {
                    "ckpt_root": ckpt_dir,
                    "dataset_dir_name": dataset_dir_name,
                    "base": base,
                    "files": found,
                }
            )

if not ckpt_candidates:
    print("=== Các thư mục ckpt tìm thấy trong /kaggle/input ===")
    all_ckpt_dirs = [p for p in INPUT.rglob("ckpt") if p.is_dir()]
    if all_ckpt_dirs:
        for p in all_ckpt_dirs:
            print(" -", p)
            try:
                for child in sorted(p.iterdir()):
                    print("    ->", child.name)
            except Exception:
                pass
    else:
        print("Không thấy thư mục nào tên ckpt.")

    raise FileNotFoundError(
        "Không tìm thấy bộ checkpoint có đủ semantic/artifact/npr. "
        "Notebook đã thử cả ckpt/sd-v1_4 và ckpt/sd-v1-4."
    )

selected_ckpt = ckpt_candidates[0]
ckpt_root = selected_ckpt["ckpt_root"]
source_ckpt_dataset_dir = selected_ckpt["dataset_dir_name"]

print("Dataset root:", data_root)
print("Checkpoint root:", ckpt_root)
print("Checkpoint dataset dir:", source_ckpt_dataset_dir)

# -------- Link training data --------
data_link = REPO / "data" / "train" / "sd-v1_4"
data_link.mkdir(parents=True, exist_ok=True)

for name in ["mscoco2017", "stable-diffusion-v1-4"]:
    link = data_link / name
    source = data_root / name
    nested = source / name

    # Tương thích với cấu trúc dataset Kaggle có thêm 1 lớp folder trùng tên.
    if nested.is_dir():
        source = nested

    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)

    link.symlink_to(source, target_is_directory=True)
    print(name, "->", source)

    for split in ["train2017", "val2017"]:
        if not (link / split).is_dir():
            raise RuntimeError(f"Thiếu {link / split}")

# -------- Copy checkpoints --------
target_ckpt = REPO / "ckpt"

if target_ckpt.exists():
    shutil.rmtree(target_ckpt)

shutil.copytree(ckpt_root, target_ckpt)

# Chuẩn hóa tên folder checkpoint về sd-v1_4 nếu nguồn dùng sd-v1-4.
if source_ckpt_dataset_dir != "sd-v1_4":
    old_dir = target_ckpt / source_ckpt_dataset_dir
    new_dir = target_ckpt / "sd-v1_4"

    if new_dir.exists():
        shutil.rmtree(new_dir)

    old_dir.rename(new_dir)

print("\n=== CHECKPOINTS ===")
for branch in ["semantic", "artifact", "npr"]:
    files = list(
        (target_ckpt / "sd-v1_4" / branch).rglob("checkpoint_best.pth")
    )

    if not files:
        raise FileNotFoundError(
            f"Thiếu checkpoint {branch} sau khi copy vào {target_ckpt / 'sd-v1_4'}"
        )

    f = files[0]
    print(branch, "->", f, f"{f.stat().st_size / 1024**2:.2f} MB")

print("\n✅ Dataset + 3 branch checkpoints: READY")


Dataset root: /kaggle/input/datasets/datnguyentrung/co-spy-v1
Checkpoint root: /kaggle/input/datasets/datnguyentrung/co-spy-v2/ckpt
Checkpoint dataset dir: sd-v1_4
mscoco2017 -> /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017
stable-diffusion-v1-4 -> /kaggle/input/datasets/datnguyentrung/co-spy-v1/stable-diffusion-v1-4/stable-diffusion-v1-4

=== CHECKPOINTS ===
semantic -> /kaggle/working/CO-SPY/ckpt/sd-v1_4/semantic/checkpoint_best.pth 0.01 MB
artifact -> /kaggle/working/CO-SPY/ckpt/sd-v1_4/artifact/checkpoint_best.pth 324.76 MB
npr -> /kaggle/working/CO-SPY/ckpt/sd-v1_4/npr/checkpoint_best.pth 5.57 MB

✅ Dataset + 3 branch checkpoints: READY


In [5]:
# 4B. Kiểm tra nhanh cây checkpoint đã chuẩn hóa

expected = REPO / "ckpt" / "sd-v1_4"

print("Expected checkpoint dir:", expected)

for branch in ["semantic", "artifact", "npr"]:
    p = expected / branch / "checkpoint_best.pth"
    print(f"{branch:8s}:", "OK" if p.exists() else "MISSING", "-", p)

assert all(
    (expected / branch / "checkpoint_best.pth").exists()
    for branch in ["semantic", "artifact", "npr"]
), "Checkpoint tree chưa đúng."

print("✅ Checkpoint tree đúng với CO-SPY.")


Expected checkpoint dir: /kaggle/working/CO-SPY/ckpt/sd-v1_4
semantic: OK - /kaggle/working/CO-SPY/ckpt/sd-v1_4/semantic/checkpoint_best.pth
artifact: OK - /kaggle/working/CO-SPY/ckpt/sd-v1_4/artifact/checkpoint_best.pth
npr     : OK - /kaggle/working/CO-SPY/ckpt/sd-v1_4/npr/checkpoint_best.pth
✅ Checkpoint tree đúng với CO-SPY.


In [6]:
# 5. Compile check + xác nhận PyTorch vẫn nhìn thấy GPU sau khi cài dependencies

import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA bị mất sau bước cài dependency. Restart Kaggle session và chạy lại."
    )

print("Training device:", f"cuda:{GPU_ID}", "-", torch.cuda.get_device_name(GPU_ID))

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        "detectors/sd_v1_4/fusion.py",
        "train.py",
    ],
    check=True,
)

print("✅ Python compile: OK")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU count: 2
Training device: cuda:0 - Tesla T4
✅ Python compile: OK


## 6. Train Fusion V3

Cell bên dưới:

- ép rõ `--gpu 0`;
- chỉ train `--mode fusion`;
- dùng `--fusion_head v3`;
- giữ các checkpoint Semantic / Artifact / NPR đã có;
- in GPU usage định kỳ trong lúc train để xác nhận VRAM và GPU Util thực sự tăng.

Nếu GPU monitor báo GPU 0 đang dùng VRAM và utilization > 0% trong các batch, thì training đang chạy bằng GPU.


In [7]:
# 6. Train Fusion V3 + GPU monitor

import subprocess
import threading
import time
import os
import sys

def gpu_status():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,utilization.gpu,temperature.gpu",
        "--format=csv,noheader",
    ]
    try:
        return subprocess.check_output(cmd, text=True).strip()
    except Exception as e:
        return f"Không đọc được nvidia-smi: {e}"

stop_monitor = threading.Event()

def monitor_gpu():
    # In ngay lúc bắt đầu, sau đó mỗi 30 giây.
    print("\n[GPU monitor - start]")
    print(gpu_status())

    while not stop_monitor.wait(30):
        print("\n[GPU monitor]")
        print(gpu_status())

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()

cmd = [
    sys.executable,
    "-u",
    "main.py",
    "--gpu", str(GPU_ID),
    "--phase", "train",
    "--mode", "fusion",
    "--train_dataset", "sd-v1_4",
    "--ckpt", "/kaggle/working/CO-SPY/ckpt",
    "--fusion_head", "v3",
    "--epochs", "5",
    "--early-stopping-patience", "2",
    "--artifact_views", "1",
    "--num-workers", "2",
    "--prefetch-factor", "2",
    "--precision", "auto",
]

print("\n=== TRAIN COMMAND ===")
print(" ".join(cmd))
print("\n=== TRAIN START ===")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

try:
    subprocess.run(cmd, check=True, env=env)
finally:
    stop_monitor.set()
    monitor_thread.join(timeout=2)

print("\n=== FINAL GPU STATUS ===")
print(gpu_status())
print("\n✅ Training process finished.")



[GPU monitor - start]

=== TRAIN COMMAND ===
/usr/bin/python3 -u main.py --gpu 0 --phase train --mode fusion --train_dataset sd-v1_4 --ckpt /kaggle/working/CO-SPY/ckpt --fusion_head v3 --epochs 5 --early-stopping-patience 2 --artifact_views 1 --num-workers 2 --prefetch-factor 2 --precision auto

=== TRAIN START ===
0, Tesla T4, 139 MiB, 15360 MiB, 0 %, 47
1, Tesla T4, 3 MiB, 15360 MiB, 0 %, 36


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]



[GPU monitor]
0, Tesla T4, 139 MiB, 15360 MiB, 0 %, 47
1, Tesla T4, 3 MiB, 15360 MiB, 0 %, 36


I0000 00:00:1795111822.644432     152 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.



[GPU monitor]
0, Tesla T4, 139 MiB, 15360 MiB, 0 %, 48
1, Tesla T4, 3 MiB, 15360 MiB, 0 %, 36


2026-11-19 18:11:08.194 | INFO     | train:__init__:299 - Trainable parameters: 14
2026-11-19 18:11:08.204 | INFO     | train:train:712 - [TRAINING START] mode=fusion | branch=artifact | dataset=sd-v1_4 | epochs=5 | batch_size=32 | seed=1024
2026-11-19 18:11:08.204 | INFO     | train:train:717 - [RUNTIME] device=cuda:0 | python=3.13.16 | torch=2.11.0+cu130 | cuda_available=True | precision=fp16 | workers=2 | prefetch=2 | pid=152
2026-11-19 18:11:08.205 | INFO     | train:train:726 - [RUNTIME] gpu=Tesla T4 | memory=14.6 GiB
2026-11-19 18:11:08.205 | INFO     | train:train:730 - [MODEL] class=CoSpyFusionDetector | architecture=detectors.sd_v1_4.fusion.CoSpyFusionDetector:b7b7fd6626f84a73bfdcd43c1bdeef232dce5fff452151feda7fa9af61874bcd
2026-11-19 18:11:08.205 | INFO     | train:train:734 - [OPTIMIZER] AdamW | lr=1.00e-03 | scheduler=none
2026-11-19 18:11:08.205 | INFO     | train:train:739 - [CHECKPOINT CONFIG] mode=auto | dir=/kaggle/working/CO-SPY/ckpt/sd-v1_4/fusion_v3 | interval=15.0m

... (9 outputs left out)

Caching fusion features (train):   1%|          | 33/5994 [02:35<6:53:07,  4.16s/it]


[GPU monitor]
0, Tesla T4, 13787 MiB, 15360 MiB, 100 %, 77
1, Tesla T4, 3 MiB, 15360 MiB, 0 %, 35


Caching fusion features (train):   1%|          | 34/5994 [02:39<6:53:16,  4.16s/it]

KeyboardInterrupt: 

In [ ]:
# 7. Liệt kê output Fusion V3

fusion_out = REPO / "ckpt" / "sd-v1_4" / "fusion_v3"

print("Fusion V3 output:", fusion_out)

if not fusion_out.exists():
    print("⚠️ Chưa thấy thư mục fusion_v3. Kiểm tra log training phía trên.")
else:
    files = [p for p in sorted(fusion_out.rglob("*")) if p.is_file()]

    if not files:
        print("⚠️ Thư mục fusion_v3 tồn tại nhưng chưa có file.")
    else:
        for p in files:
            print(p, f"{p.stat().st_size / 1024**2:.2f} MB")

print("\nHoàn tất. Nếu checkpoint đã xuất hiện, hãy Save Version trên Kaggle để giữ output.")
